# Cardiovascular Exercise Physiology

**Clinical Application:** Exercise testing, fitness assessment, and cardiac rehabilitation

**Learning Objectives:**
1. Understand integrated cardiovascular responses to exercise
2. Simulate autonomic adaptations during physical activity
3. Analyze heart rate recovery as a prognostic marker
4. Interpret exercise capacity in health and disease

**Clinical Relevance:**
- Exercise stress testing for CAD diagnosis
- Cardiopulmonary exercise testing (CPET) for heart failure
- Cardiac rehabilitation prescription
- Sudden cardiac death risk stratification

**Physiological Background:**

During exercise, the cardiovascular system must rapidly increase oxygen delivery to working muscles through coordinated responses:

**Central Command:**
- CNS anticipatory signals (cortical motor areas)
- Immediate ↓ parasympathetic withdrawal
- Gradual ↑ sympathetic activation

**Peripheral Feedback:**
- Muscle mechanoreceptors (type III afferents)
- Muscle metaboreceptors (type IV - lactate, H+, K+)
- Arterial baroreceptors (BP regulation)
- Chemoreceptors (CO2, O2)

**Hemodynamic Changes:**
- Heart rate: 60 → 180+ bpm (3x increase)
- Stroke volume: 70 → 120 mL (1.7x increase)
- Cardiac output: 5 → 25+ L/min (5x increase)
- Blood pressure: MAP 93 → 120+ mmHg

**References:**
- Rowell LB (1993) Human Cardiovascular Control - Oxford University Press
- Mitchell JH (2012) Neural circulatory control during exercise. *Compr Physiol* 2:2325-2380
- Cole CR et al. (1999) Heart-rate recovery immediately after exercise as a predictor of mortality. *NEJM* 341:1351-1357

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.autonomic.autonomic_nervous_system import (
    AutonomicNervousSystem, 
    AutonomicParameters,
    AutonomicState
)
from src.validation.benchmarks import PhysiologicalBenchmarks

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 12)
print("✓ Imports successful")

## Part 1: Progressive Exercise Test (Bruce Protocol)

The Bruce treadmill protocol is the most widely used exercise test, with 7 stages of 3 minutes each, progressively increasing speed and incline.

**Bruce Protocol Stages:**
- Stage 1: 1.7 mph, 10% grade (4.7 METs)
- Stage 2: 2.5 mph, 12% grade (7.0 METs)
- Stage 3: 3.4 mph, 14% grade (10.1 METs)
- Stage 4: 4.2 mph, 16% grade (12.9 METs)
- Stage 5: 5.0 mph, 18% grade (15.0 METs)

In [ ]:
def simulate_bruce_protocol(ans, max_duration=15.0, dt=0.01):
    """
    Simulate Bruce protocol exercise test.
    
    Args:
        ans: AutonomicNervousSystem instance
        max_duration: Maximum test duration (minutes)
        dt: Time step (seconds)
    
    Returns:
        Dictionary with time series data
    """
    results = {
        'time_min': [],
        'stage': [],
        'mets': [],
        'central_command': [],
        'pressure': [],
        'heart_rate': [],
        'cardiac_output': [],
        'vagal_tone': [],
        'sympathetic_tone': [],
    }
    
    # Bruce protocol stages (MET levels)
    stage_times = [0, 3, 6, 9, 12]  # minutes
    stage_mets = [1.0, 4.7, 7.0, 10.1, 12.9]  # metabolic equivalents
    
    baseline_pressure = 93.0  # mmHg
    baseline_hr = 70.0  # bpm
    baseline_sv = 70.0  # mL
    
    t = 0.0
    duration_sec = max_duration * 60.0
    
    while t < duration_sec:
        t_min = t / 60.0
        
        # Determine current stage and MET level
        stage = 0
        for i, stage_time in enumerate(stage_times):
            if t_min >= stage_time:
                stage = i
        
        current_mets = stage_mets[stage]
        
        # Central command scales with MET level (0-1)
        central_command = min(1.0, (current_mets - 1.0) / 14.0)
        
        # Blood pressure increases with exercise
        # Systolic increases more than diastolic (pulse pressure widens)
        pressure_increase = 20.0 * central_command
        pressure = baseline_pressure + pressure_increase
        
        # Update autonomic nervous system
        state = ans.update(
            pressure=pressure,
            dt=dt,
            t=t,
            central_command=central_command,
        )
        
        # Compute cardiovascular parameters
        hr = ans.get_heart_rate(intrinsic_hr=105.0)
        
        # Stroke volume increases then plateaus (~50% increase max)
        sv_increase = 1.0 + 0.5 * min(1.0, central_command * 2.0)
        sv = baseline_sv * sv_increase
        
        # Cardiac output = HR × SV (mL/min → L/min)
        co = (hr * sv) / 1000.0
        
        # Store results (subsample every 100 timesteps for efficiency)
        if int(t / dt) % 100 == 0:
            results['time_min'].append(t_min)
            results['stage'].append(stage)
            results['mets'].append(current_mets)
            results['central_command'].append(central_command)
            results['pressure'].append(pressure)
            results['heart_rate'].append(hr)
            results['cardiac_output'].append(co)
            results['vagal_tone'].append(state.vagal_tone)
            results['sympathetic_tone'].append(state.sympathetic_tone)
        
        t += dt
    
    return results

# Create normal autonomic system
ans_healthy = AutonomicNervousSystem()

print("="*60)
print("BRUCE PROTOCOL EXERCISE TEST - HEALTHY INDIVIDUAL")
print("="*60)
print("\nSimulating progressive exercise test...\n")

results_bruce = simulate_bruce_protocol(ans_healthy, max_duration=15.0)

# Convert to numpy arrays
times = np.array(results_bruce['time_min'])
stages = np.array(results_bruce['stage'])
mets = np.array(results_bruce['mets'])
hr = np.array(results_bruce['heart_rate'])
co = np.array(results_bruce['cardiac_output'])
pressure = np.array(results_bruce['pressure'])
vagal = np.array(results_bruce['vagal_tone'])
sympathetic = np.array(results_bruce['sympathetic_tone'])

# Create comprehensive visualization
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)

# Stage color coding
stage_colors = ['lightgray', 'lightblue', 'lightyellow', 'lightcoral', 'lightpink']
stage_labels = ['Rest', 'Stage 1\n(4.7 METs)', 'Stage 2\n(7.0 METs)', 
               'Stage 3\n(10.1 METs)', 'Stage 4\n(12.9 METs)']

for ax in axes:
    for i in range(5):
        stage_mask = stages == i
        if np.any(stage_mask):
            t_start = times[stage_mask][0]
            t_end = times[stage_mask][-1]
            ax.axvspan(t_start, t_end, alpha=0.15, color=stage_colors[i])

# Plot 1: Heart Rate
axes[0].plot(times, hr, 'r-', linewidth=2.5)
axes[0].axhline(70, color='gray', linestyle='--', alpha=0.5, label='Baseline')
# Age-predicted max HR (220 - age), assuming age 40
axes[0].axhline(180, color='orange', linestyle='--', alpha=0.7, label='Age-predicted max (220-40)')
axes[0].set_ylabel('Heart Rate\n(bpm)', fontsize=12, fontweight='bold')
axes[0].set_title('Bruce Protocol Exercise Test - Cardiovascular Response', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([50, 200])

# Plot 2: Cardiac Output
axes[1].plot(times, co, 'b-', linewidth=2.5)
axes[1].axhline(5.0, color='gray', linestyle='--', alpha=0.5, label='Resting CO')
axes[1].set_ylabel('Cardiac Output\n(L/min)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3)

# Plot 3: Blood Pressure
axes[2].plot(times, pressure, 'purple', linewidth=2.5)
axes[2].axhline(93, color='gray', linestyle='--', alpha=0.5, label='Resting MAP')
axes[2].axhline(120, color='red', linestyle='--', alpha=0.7, label='Excessive BP response')
axes[2].set_ylabel('Mean Arterial\nPressure (mmHg)', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10, loc='upper left')
axes[2].grid(True, alpha=0.3)

# Plot 4: Autonomic Balance
axes[3].plot(times, vagal, 'g-', linewidth=2.5, label='Vagal (Parasympathetic)', alpha=0.8)
axes[3].plot(times, sympathetic, 'orange', linewidth=2.5, label='Sympathetic', alpha=0.8)
axes[3].set_ylabel('Autonomic\nTone (0-1)', fontsize=12, fontweight='bold')
axes[3].legend(fontsize=10, loc='right')
axes[3].set_ylim([0, 1.0])
axes[3].grid(True, alpha=0.3)

# Plot 5: Stage Indicator
axes[4].plot(times, mets, 'k-', linewidth=3)
axes[4].set_xlabel('Time (minutes)', fontsize=12, fontweight='bold')
axes[4].set_ylabel('Workload\n(METs)', fontsize=12, fontweight='bold')
axes[4].grid(True, alpha=0.3)
axes[4].set_ylim([0, 15])

# Add stage labels
for i in range(5):
    stage_mask = stages == i
    if np.any(stage_mask):
        t_mid = np.mean(times[stage_mask])
        axes[4].text(t_mid, mets[stage_mask][0] + 0.5, stage_labels[i],
                    ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Print test summary
print("\n" + "="*60)
print("EXERCISE TEST RESULTS")
print("="*60)
print(f"\nResting:")
print(f"  Heart Rate: {hr[0]:.0f} bpm")
print(f"  Cardiac Output: {co[0]:.1f} L/min")
print(f"  Blood Pressure: {pressure[0]:.0f} mmHg")

print(f"\nPeak Exercise (Stage 4):")
print(f"  Heart Rate: {hr[-1]:.0f} bpm ({(hr[-1]/hr[0]):.1f}x increase)")
print(f"  Cardiac Output: {co[-1]:.1f} L/min ({(co[-1]/co[0]):.1f}x increase)")
print(f"  Blood Pressure: {pressure[-1]:.0f} mmHg (+{pressure[-1]-pressure[0]:.0f} mmHg)")

print(f"\nChronotropic Response:")
hr_reserve = ((hr[-1] - hr[0]) / (180 - hr[0])) * 100  # Percent of HR reserve achieved
print(f"  HR Reserve Used: {hr_reserve:.1f}%")
if hr_reserve >= 85:
    print(f"  Interpretation: Excellent chronotropic competence")
elif hr_reserve >= 70:
    print(f"  Interpretation: Good chronotropic response")
else:
    print(f"  Interpretation: Chronotropic incompetence (consider beta-blocker effect or sinus node dysfunction)")

print("\n" + "="*60)

## Part 2: Heart Rate Recovery - Prognostic Marker

Heart rate recovery (HRR) after exercise is a powerful prognostic marker. Delayed recovery indicates autonomic dysfunction and is associated with increased mortality.

**Cole et al. (1999) NEJM Study:**
- HRR <12 bpm at 1 minute post-exercise → 2x mortality risk
- HRR <22 bpm at 2 minutes → Increased risk
- Mechanism: Impaired parasympathetic reactivation

In [ ]:
def simulate_exercise_recovery(ans, exercise_duration=10.0, recovery_duration=10.0, dt=0.01):
    """
    Simulate exercise followed by recovery.
    """
    results = {
        'time_min': [],
        'phase': [],  # 'rest', 'exercise', 'recovery'
        'heart_rate': [],
        'vagal_tone': [],
        'sympathetic_tone': [],
    }
    
    total_duration = 3.0 + exercise_duration + recovery_duration  # 3 min rest + exercise + recovery
    rest_end = 3.0
    exercise_end = rest_end + exercise_duration
    
    t = 0.0
    while t / 60.0 < total_duration:
        t_min = t / 60.0
        
        # Determine phase and central command
        if t_min < rest_end:
            phase = 'rest'
            central_command = 0.0
            pressure = 93.0
        elif t_min < exercise_end:
            phase = 'exercise'
            # Ramp up to moderate exercise (8 METs ~ 0.5 central command)
            time_in_exercise = t_min - rest_end
            central_command = min(0.5, time_in_exercise / 2.0)  # Ramp up over 2 min
            pressure = 93.0 + 15.0 * central_command
        else:
            phase = 'recovery'
            time_in_recovery = t_min - exercise_end
            # Central command decays exponentially
            central_command = 0.5 * np.exp(-time_in_recovery / 2.0)
            pressure = 93.0 + 15.0 * central_command
        
        # Update autonomic system
        state = ans.update(
            pressure=pressure,
            dt=dt,
            t=t,
            central_command=central_command,
        )
        
        hr = ans.get_heart_rate(intrinsic_hr=105.0)
        
        # Store results (subsample)
        if int(t / dt) % 100 == 0:
            results['time_min'].append(t_min)
            results['phase'].append(phase)
            results['heart_rate'].append(hr)
            results['vagal_tone'].append(state.vagal_tone)
            results['sympathetic_tone'].append(state.sympathetic_tone)
        
        t += dt
    
    return results

# Simulate healthy individual
ans_healthy = AutonomicNervousSystem()
results_healthy = simulate_exercise_recovery(ans_healthy)

# Simulate individual with autonomic dysfunction (e.g., post-MI, diabetes)
from src.autonomic.baroreflex import BaroreflexParameters

impaired_params = AutonomicParameters(
    baseline_vagal_tone=0.4,  # Reduced
    vagal_time_constant=2.0,  # Slower vagal reactivation (normal 0.5s)
)

ans_impaired = AutonomicNervousSystem(params=impaired_params)
results_impaired = simulate_exercise_recovery(ans_impaired)

# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

times_healthy = np.array(results_healthy['time_min'])
hr_healthy = np.array(results_healthy['heart_rate'])
vagal_healthy = np.array(results_healthy['vagal_tone'])

times_impaired = np.array(results_impaired['time_min'])
hr_impaired = np.array(results_impaired['heart_rate'])
vagal_impaired = np.array(results_impaired['vagal_tone'])

# Mark phases
for ax in axes:
    ax.axvspan(0, 3, alpha=0.1, color='gray', label='Rest')
    ax.axvspan(3, 13, alpha=0.1, color='yellow', label='Exercise')
    ax.axvspan(13, 23, alpha=0.1, color='lightblue', label='Recovery')
    ax.axvline(13, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Plot 1: Heart Rate Comparison
axes[0].plot(times_healthy, hr_healthy, 'b-', linewidth=2.5, label='Healthy', alpha=0.8)
axes[0].plot(times_impaired, hr_impaired, 'r--', linewidth=2.5, label='Autonomic Dysfunction', alpha=0.8)
axes[0].set_ylabel('Heart Rate (bpm)', fontsize=12, fontweight='bold')
axes[0].set_title('Heart Rate Recovery - Prognostic Assessment', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11, loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: Vagal Reactivation
axes[1].plot(times_healthy, vagal_healthy, 'g-', linewidth=2.5, label='Healthy', alpha=0.8)
axes[1].plot(times_impaired, vagal_impaired, 'orange', linestyle='--', linewidth=2.5, 
            label='Autonomic Dysfunction', alpha=0.8)
axes[1].set_ylabel('Vagal Tone (0-1)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11, loc='right')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

# Plot 3: HR Recovery Rate (zoomed to recovery phase)
# Find exercise end index
ex_end_idx_h = np.argmin(np.abs(times_healthy - 13.0))
ex_end_idx_i = np.argmin(np.abs(times_impaired - 13.0))

recovery_times_h = times_healthy[ex_end_idx_h:] - 13.0
recovery_hr_h = hr_healthy[ex_end_idx_h:]
recovery_times_i = times_impaired[ex_end_idx_i:] - 13.0
recovery_hr_i = hr_impaired[ex_end_idx_i:]

axes[2].plot(recovery_times_h, recovery_hr_h - recovery_hr_h[0], 'b-', linewidth=2.5, 
            label='Healthy', alpha=0.8)
axes[2].plot(recovery_times_i, recovery_hr_i - recovery_hr_i[0], 'r--', linewidth=2.5, 
            label='Autonomic Dysfunction', alpha=0.8)
axes[2].axhline(-12, color='orange', linestyle='--', alpha=0.7, label='1-min threshold (Cole)')
axes[2].axvline(1.0, color='gray', linestyle=':', alpha=0.5)
axes[2].axvline(2.0, color='gray', linestyle=':', alpha=0.5)
axes[2].set_xlabel('Time After Exercise Cessation (minutes)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('ΔHR from Peak (bpm)', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=11, loc='lower right')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([0, 5])

plt.tight_layout()
plt.show()

# Calculate HRR metrics
idx_1min_h = np.argmin(np.abs(recovery_times_h - 1.0))
idx_1min_i = np.argmin(np.abs(recovery_times_i - 1.0))
idx_2min_h = np.argmin(np.abs(recovery_times_h - 2.0))
idx_2min_i = np.argmin(np.abs(recovery_times_i - 2.0))

hrr_1min_h = recovery_hr_h[0] - recovery_hr_h[idx_1min_h]
hrr_1min_i = recovery_hr_i[0] - recovery_hr_i[idx_1min_i]
hrr_2min_h = recovery_hr_h[0] - recovery_hr_h[idx_2min_h]
hrr_2min_i = recovery_hr_i[0] - recovery_hr_i[idx_2min_i]

print("\n" + "="*60)
print("HEART RATE RECOVERY ANALYSIS (Cole et al. 1999)")
print("="*60)

print(f"\nHealthy Individual:")
print(f"  Peak Exercise HR: {recovery_hr_h[0]:.0f} bpm")
print(f"  HRR at 1 minute: {hrr_1min_h:.0f} bpm {'✓ Normal' if hrr_1min_h >= 12 else '⚠️  Abnormal'}")
print(f"  HRR at 2 minutes: {hrr_2min_h:.0f} bpm {'✓ Normal' if hrr_2min_h >= 22 else '⚠️  Abnormal'}")
print(f"  Mortality Risk: Low")

print(f"\nAutonomic Dysfunction:")
print(f"  Peak Exercise HR: {recovery_hr_i[0]:.0f} bpm")
print(f"  HRR at 1 minute: {hrr_1min_i:.0f} bpm {'✓ Normal' if hrr_1min_i >= 12 else '⚠️  Abnormal'}")
print(f"  HRR at 2 minutes: {hrr_2min_i:.0f} bpm {'✓ Normal' if hrr_2min_i >= 22 else '⚠️  Abnormal'}")
print(f"  Mortality Risk: {'High (2-4x)' if hrr_1min_i < 12 else 'Moderate'}")

print(f"\nClinical Interpretation:")
if hrr_1min_i < 12:
    print(f"  - Impaired parasympathetic reactivation")
    print(f"  - Increased sudden cardiac death risk")
    print(f"  - Consider: Cardiac autonomic neuropathy workup")
    print(f"  - Recommend: Cardiac rehabilitation, aerobic training")
    print(f"  - Follow-up: Repeat testing after 3-6 months")

print("\n" + "="*60)
print("REFERENCE: Cole CR et al. (1999) NEJM 341:1351-1357")
print("  - Study: 2,428 patients, 6-year follow-up")
print("  - HRR <12 bpm → RR 2.0 for all-cause mortality")
print("  - Independent of perfusion defects, coronary disease severity")
print("="*60)

## Part 3: Exercise Prescription - Karvonen Formula

Evidence-based exercise prescription for cardiac rehabilitation and fitness improvement.

**Karvonen Formula:**
Target HR = [(HRmax - HRrest) × %Intensity] + HRrest

**ACSM Guidelines:**
- Moderate intensity: 40-60% HRR
- Vigorous intensity: 60-85% HRR
- Cardiac rehab (Phase II): 40-70% HRR

In [ ]:
def calculate_exercise_prescription(age, resting_hr, fitness_level='moderate'):
    """
    Calculate personalized exercise prescription using Karvonen formula.
    """
    # Age-predicted maximum heart rate
    hr_max = 220 - age
    
    # Heart rate reserve
    hr_reserve = hr_max - resting_hr
    
    # Training zones
    zones = {
        'recovery': (0.30, 0.40),
        'aerobic_base': (0.40, 0.60),
        'tempo': (0.60, 0.70),
        'threshold': (0.70, 0.85),
        'vo2max': (0.85, 1.00),
    }
    
    prescription = {}
    for zone_name, (low, high) in zones.items():
        hr_low = (hr_reserve * low) + resting_hr
        hr_high = (hr_reserve * high) + resting_hr
        prescription[zone_name] = (hr_low, hr_high)
    
    return hr_max, hr_reserve, prescription

# Example prescriptions for different populations
patients = [
    {'name': 'Healthy Adult (40 yo)', 'age': 40, 'resting_hr': 65},
    {'name': 'Post-MI Patient (55 yo)', 'age': 55, 'resting_hr': 75},
    {'name': 'Heart Failure (65 yo)', 'age': 65, 'resting_hr': 85},
    {'name': 'Athlete (30 yo)', 'age': 30, 'resting_hr': 45},
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

zone_colors = ['lightgreen', 'yellow', 'orange', 'red', 'darkred']
zone_labels = ['Recovery', 'Aerobic Base', 'Tempo', 'Threshold', 'VO2max']

for idx, patient in enumerate(patients):
    ax = axes[idx]
    
    hr_max, hr_reserve, zones_rx = calculate_exercise_prescription(
        patient['age'], 
        patient['resting_hr']
    )
    
    # Create bar chart of training zones
    zone_names = list(zones_rx.keys())
    y_pos = np.arange(len(zone_names))
    
    for i, zone_name in enumerate(zone_names):
        hr_low, hr_high = zones_rx[zone_name]
        width = hr_high - hr_low
        ax.barh(i, width, left=hr_low, height=0.7, 
               color=zone_colors[i], edgecolor='black', linewidth=1.5, alpha=0.8)
        # Add HR range labels
        ax.text(hr_low + width/2, i, f'{hr_low:.0f}-{hr_high:.0f}', 
               ha='center', va='center', fontsize=10, fontweight='bold')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(zone_labels, fontsize=10)
    ax.set_xlabel('Heart Rate (bpm)', fontsize=11, fontweight='bold')
    ax.set_title(f"{patient['name']}\n" + 
                f"Resting HR: {patient['resting_hr']} | Max HR: {hr_max:.0f} | Reserve: {hr_reserve:.0f}",
                fontsize=11, fontweight='bold')
    ax.axvline(patient['resting_hr'], color='blue', linestyle='--', linewidth=2, 
              alpha=0.7, label='Resting')
    ax.axvline(hr_max, color='red', linestyle='--', linewidth=2, 
              alpha=0.7, label='Maximum')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim([0, 200])

plt.suptitle('Personalized Exercise Prescription (Karvonen Method)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print detailed recommendations
print("\n" + "="*60)
print("EXERCISE PRESCRIPTION RECOMMENDATIONS")
print("="*60)

for patient in patients:
    hr_max, hr_reserve, zones_rx = calculate_exercise_prescription(
        patient['age'], 
        patient['resting_hr']
    )
    
    print(f"\n{patient['name']}:")
    print(f"  Resting HR: {patient['resting_hr']} bpm")
    print(f"  Max HR (220-age): {hr_max:.0f} bpm")
    print(f"  HR Reserve: {hr_reserve:.0f} bpm")
    print(f"\n  Recommended Training Zones:")
    
    # Give specific recommendations
    if 'Post-MI' in patient['name']:
        target_zone = zones_rx['aerobic_base']
        print(f"    PRIMARY: Aerobic Base {target_zone[0]:.0f}-{target_zone[1]:.0f} bpm")
        print(f"    Duration: 20-40 minutes, 3-5 days/week")
        print(f"    Modality: Walking, cycling, swimming")
        print(f"    Progression: Increase duration before intensity")
    elif 'Heart Failure' in patient['name']:
        target_zone = zones_rx['recovery']
        alt_zone = zones_rx['aerobic_base']
        print(f"    START: Recovery {target_zone[0]:.0f}-{target_zone[1]:.0f} bpm")
        print(f"    PROGRESS TO: Aerobic {alt_zone[0]:.0f}-{alt_zone[1]:.0f} bpm")
        print(f"    Duration: 10-30 minutes, 5 days/week")
        print(f"    Caution: Monitor symptoms (dyspnea, fatigue)")
    elif 'Athlete' in patient['name']:
        print(f"    BASE: {zones_rx['aerobic_base'][0]:.0f}-{zones_rx['aerobic_base'][1]:.0f} bpm (80% of volume)")
        print(f"    TEMPO: {zones_rx['tempo'][0]:.0f}-{zones_rx['tempo'][1]:.0f} bpm (15% of volume)")
        print(f"    THRESHOLD: {zones_rx['threshold'][0]:.0f}-{zones_rx['threshold'][1]:.0f} bpm (5% of volume)")
        print(f"    80/20 polarized training model")
    else:
        target_zone = zones_rx['aerobic_base']
        print(f"    PRIMARY: Aerobic Base {target_zone[0]:.0f}-{target_zone[1]:.0f} bpm")
        print(f"    Duration: 30-60 minutes, 5 days/week")
        print(f"    ACSM recommendation: 150 min/week moderate intensity")

print("\n" + "="*60)
print("REFERENCES:")
print("  - Karvonen MJ et al. (1957) Annales Medicinae Experimentalis")
print("  - ACSM Guidelines (2022) Exercise Prescription")
print("  - AHA Scientific Statement on Exercise Standards (2013)")
print("="*60)

## Summary

### Key Concepts

**1. Cardiovascular Responses to Exercise:**
- Coordinated autonomic responses (vagal withdrawal + sympathetic activation)
- Heart rate and cardiac output increase proportionally to workload
- Stroke volume increases 40-50% then plateaus
- Blood pressure increases (systolic more than diastolic)

**2. Heart Rate Recovery (HRR):**
- Powerful prognostic marker (Cole et al. 1999)
- HRR <12 bpm at 1 minute → 2x mortality risk
- Reflects parasympathetic reactivation
- Improves with training

**3. Exercise Prescription:**
- Karvonen formula for individualized targets
- Heart rate reserve method most accurate
- Different populations require different intensity zones
- Progressive overload principle

**4. Clinical Applications:**
- Exercise stress testing for CAD diagnosis
- Cardiac rehabilitation (post-MI, heart failure)
- Risk stratification (chronotropic incompetence, HRR)
- Athletic performance optimization

### Exercise Training Adaptations

**Cardiovascular:**
- ↑ Stroke volume (eccentric hypertrophy)
- ↓ Resting heart rate
- ↑ Maximum cardiac output
- ↑ Capillary density

**Autonomic:**
- ↑ Parasympathetic tone at rest
- ↓ Sympathetic tone at rest
- ↑ Heart rate variability
- Faster HR recovery

---
© 2025 Multi-Heart-Model Project | MIT License